In [17]:
from mlflow.models import validate_serving_input
from mlflow.models import convert_input_example_to_serving_input
import json
 

In [15]:
import mlflow
from mlflow.entities import ViewType

# 1. Ambil client MLflow
client = mlflow.MlflowClient()

# 2. Cari eksperimen yang statusnya HANYA yang sudah dihapus (DELETED_ONLY)
deleted_experiments = client.search_experiments(view_type=ViewType.DELETED_ONLY)

print("--- DAFTAR EKSPERIMEN DI FOLDER SAMPAH ---")
if not deleted_experiments:
    print("Folder sampah kosong!")
else:
    for exp in deleted_experiments:
        print(f"Nama Eksperimen : {exp.name}")
        print(f"Experiment ID   : {exp.experiment_id}")
        print(f"Lokasi Folder   : {exp.artifact_location}")
        print("-" * 40)

--- DAFTAR EKSPERIMEN DI FOLDER SAMPAH ---
Nama Eksperimen : Latihan Credit Scoring
Experiment ID   : 1
Lokasi Folder   : mlflow-artifacts:/1
----------------------------------------
Nama Eksperimen : Default
Experiment ID   : 0
Lokasi Folder   : mlflow-artifacts:/0
----------------------------------------


In [18]:
import mlflow

# Ganti dengan run_id kamu yang sebenarnya
print("Tracking URI saat ini:", mlflow.get_tracking_uri())

try:
    run = mlflow.get_run("f7708bebd31849c991f5bc749870049d")
    print("Run ditemukan! Status:", run.info.status)
except Exception as e:
    print("Error nih, emang gak ketemu Run-nya:", e)

Tracking URI saat ini: http://localhost:5000
Run ditemukan! Status: FINISHED


In [39]:
import mlflow

run_id = "1d33c24e3e7e4235b3a8fdb59198a21b"
model_uri = f"runs:/{run_id}/model/serving_input_example.json"

# List semua file yang ada di dalam folder model
client = mlflow.MlflowClient()
artifacts = client.list_artifacts(run_id, path="model")

print("Daftar file yang ada di artifact model:")
for artifact in artifacts:
    print("-", artifact.path)

Daftar file yang ada di artifact model:
- model/MLmodel
- model/conda.yaml
- model/model.pkl
- model/python_env.yaml
- model/requirements.txt


In [37]:
import mlflow

run_id = "1d33c24e3e7e4235b3a8fdb59198a21b"

client = mlflow.MlflowClient()
# Kosongkan bagian path-nya agar kelihatan semua struktur folders
artifacts = client.list_artifacts(run_id) 

print("Struktur Folder Artifact Selengkapnya:")
for artifact in artifacts:
    print("- Is Directory:", artifact.is_dir, "| Path:", artifact.path)

Struktur Folder Artifact Selengkapnya:
- Is Directory: False | Path: estimator.html
- Is Directory: False | Path: metric_info.json
- Is Directory: False | Path: training_confusion_matrix.png


In [48]:
import mlflow

run_id = "1d33c24e3e7e4235b3a8fdb59198a21b"
client = mlflow.MlflowClient()

# Cek root folder artifact-nya dulu
print("--- DAFTAR ROOT ARTIFACT ---")
for art in client.list_artifacts(run_id):
    print(f"- Path: {art.path} | Is Dir: {art.is_dir}")

# Cek folder model yang dicurigai duplikat
print("\n--- ISI FOLDER 'model' ---")
try:
    for art in client.list_artifacts(run_id, path="model"):
        print(f"- {art.path}")
except Exception as e:
    print("Folder 'model' tidak ditemukan:", e)

--- DAFTAR ROOT ARTIFACT ---
- Path: estimator.html | Is Dir: False
- Path: metric_info.json | Is Dir: False
- Path: training_confusion_matrix.png | Is Dir: False

--- ISI FOLDER 'model' ---
- model/MLmodel
- model/conda.yaml
- model/model.pkl
- model/python_env.yaml
- model/requirements.txt


In [47]:
run_id = "1d33c24e3e7e4235b3a8fdb59198a21b"

model_uri = f'runs:/{run_id}/model'
data = f'runs:/{run_id}/model/serving_input_example.json'

# Catatan: Karena file JSON berada di dalam artifact MLflow, 
# kita perlu mendownload atau mengambil path lokalnya terlebih dahulu agar bisa dibuka dengan open()
#local_path = mlflow.artifacts.download_artifacts(artifact_uri=data)

local_path = mlflow.artifacts.download_artifacts(artifact_uri=data)
 
# Membuka dan membaca file JSON 
with open(local_path, "r") as file: 
    serving_payload = json.load(file) # Memuat JSON menjadi dictionary
 
# Validate the serving payload works on the model
validate_serving_input(model_uri, serving_payload)

MlflowException: Failed to download artifacts from path 'serving_input_example.json', please ensure that the path is correct.